Anexo A, apartado A.2 · Adaptadores: configuración común de entrenamiento.

Cuaderno con el que se entrenó LoRA-ruido en una GPU A100 de 80 GB de Google Colab, con el entrenador y
la configuración comunes, y se calculó su verosimilitud por sesión sobre el corpus de evaluación.


# Stage 7.5.1 — LoRA-noise training (Google Colab variant)

Entrena LoRA-noise adapter (matching Centaur paper LoRA config) sobre corpus sintético de noise prompts. Replicates el control 2-way del [revisión interna] [revisión interna] LoRA defense.

**Path**: **canonical A100 80GB** (`max_seq_length=32768` from `lora_config.yaml`).
**Compute**: ~20h A100 80GB. Cabe en 1 session Colab Pro+ (24h limit).
**Cost**: ~$8-10 equivalente.
**Requires**: Colab Pro+ con A100 80GB, HF token, Drive con ≥30GB libre.

**Reduced A100 40GB path** (opt-in, NO default):
- En Cell 2: añadir `--reduced-seq-plan` al check_colab_env.py.
- En Cell 5: añadir `--max-seq-length 16384` al `train_lora_noise.py` command.
- WARNING: reducing puede dropear answer spans `<<X>>` en prompts long-context; verify pre-launch.

Pipeline:
1. Setup Colab env
2. Generate noise corpus (CPU, ~10 min)
3. Train LoRA-noise (A100, ~20h)
4. 2-way eval Centaur vs LoRA-noise sobre IGT data (A100, ~1h)


## Cell 1: Colab env setup

In [ ]:
# Clone repo + run colab_setup.py
# Si el repo es privado, requiere GH_TOKEN en Colab Secrets (🔑 sidebar):
#   Name: GH_TOKEN, Value: PAT con `Contents: Read-only` para este repo, Notebook access ON.
# Si es public, GH_TOKEN se ignora (clone anónimo).
from google.colab import userdata
import os
gh_token = None
try:
    gh_token = userdata.get('GH_TOKEN')
except Exception:
    pass

if not os.path.exists('/content/ai-system-lab'):
    if gh_token:
        !git clone <REPO> /content/ai-system-lab
    else:
        !git clone <REPO> /content/ai-system-lab
else:
    !cd /content/ai-system-lab && git pull

assert os.path.exists('/content/ai-system-lab/tesis'), 'Clone failed — revisa GH_TOKEN o repo visibility'
!python /content/ai-system-lab/tesis/data_analyses/llm_evaluation/paper_01_igt/launchers/colab/colab_setup.py


## Cell 2: HF token + pre-flight

In [ ]:
# [revisión interna] polish post-launch: load HF_TOKEN directly into notebook kernel
# (colab_setup.py runs as subprocess; its os.environ does NOT persist al kernel)
import os
if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        tok = userdata.get('HF_TOKEN')
        if tok:
            os.environ['HF_TOKEN'] = tok
    except Exception as e:
        print(f'WARN: could not read Colab Secret: {e}')
assert os.environ.get('HF_TOKEN'), 'Set HF_TOKEN in Colab Secrets (🔑 sidebar) + toggle Notebook access ON'
%cd /content/ai-system-lab
!python tesis/data_analyses/llm_evaluation/paper_01_igt/launchers/colab/check_colab_env.py --stage 7.5.1
!python tesis/data_analyses/llm_evaluation/paper_01_igt/preflight/check_manifests_integrity.py --stage 7.5.1


## Cell 3: Generate noise corpus (CPU, ~10 min)

In [ ]:
# 100k tokens synthetic noise prompts con estructura superficial Psych-101
import os
os.makedirs('/content/drive/MyDrive/paper_01_igt_results/stage_7_5_1', exist_ok=True)

%cd tesis/data_analyses/llm_evaluation/paper_01_igt/stage_7_5_1_lora_noise/
!python generate_noise_corpus.py \
    --output /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/corpus_noise.jsonl \
    --tokens 100000 --seed 20260506

# Verify
!head -1 /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/corpus_noise.jsonl | python3 -m json.tool

## Cell 4: Dry-run config validation (1 min)

In [ ]:
# Validate LoRA config matches paper Centaur before spending GPU hours
!python train_lora_noise.py \
    --corpus /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/corpus_noise.jsonl \
    --output_adapter /tmp/dummy_dry \
    --config lora_config.yaml \
    --dry_run

## Cell 5: Train LoRA-noise (~20h on A100 80GB)

In [ ]:
# Production training run. Checkpoints to Drive cada 500 steps.
!python train_lora_noise.py \
    --corpus /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/corpus_noise.jsonl \
    --output_adapter /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/lora_noise_adapter \
    --config lora_config.yaml

# Si la session se cae a mid-run, re-run con --resume_from_checkpoint:
# !python train_lora_noise.py --corpus ... --output_adapter ... --config lora_config.yaml \
#     --resume_from_checkpoint /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/lora_noise_adapter/_training_logs/checkpoint-N

## Cell 6: 2-way eval Centaur vs LoRA-noise (~1h on A100)

In [ ]:
!python eval_2way.py \
    --centaur-adapter marcelbinz/Llama-3.1-Centaur-70B-adapter \
    --noise-adapter /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/lora_noise_adapter \
    --igt-data /content/ai-system-lab/tesis/data_analyses/llm_evaluation/paper_01_igt/manifests/igt_paper1_eval_data.json \
    --output /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/

# Display results
!cat /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/eval_2way_stats.json | python3 -m json.tool

## Done

Outputs en `/content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/`:
- `lora_noise_adapter/` — PEFT adapter (~50MB)
- `nll_centaur.json`, `nll_lora_noise.json` — per-prompt NLL en IGT eval
- `eval_2way_stats.json` — paired stats + partial H4 verdict

**Next**: Stage 7.5.2 (LoRA-irrelevant + 3-way eval) en otro notebook.


In [ ]:
# Comprimir y bajar
!zip -r /content/stage_7_5_1.zip /content/drive/MyDrive/paper_01_igt_results/stage_7_5_1/
from google.colab import files
files.download('/content/stage_7_5_1.zip')
